# 🎹 Real-World One-to-Many Piano Generation: MAESTRO v3
### Comparative Deep Learning Evaluation — LSTM vs GRU vs TCN

**Research question:** How do three different sequence decoders compare when generating piano note sequences from **one starting note**?

**Real dataset:** [MAESTRO v3.0.0](https://magenta.tensorflow.org/datasets/maestro), recordings of real piano performances, **not a synthetic dataset**. We use the official **MIDI-only** archive (~56 MB) and its **composition-disjoint train/validation/test splits**.

**One-to-many at inference:** one real seed MIDI note → **32 newly generated note pitches**. During supervised training, we use **teacher forcing**: the true previous notes are supplied to the decoder. During the demo, the model only gets the one seed and feeds back **its own predictions**.

**Important limitations:** We simplify polyphonic piano MIDI into an ordered stream of note-on pitches, and synthesize demonstration audio. This is a real-data *pitch-continuation experiment*, not a full piano performance generator (timing, duration, dynamics, and polyphony are not predicted). A single seed does not uniquely determine the future notes. Test accuracy below is evaluated with teacher forcing; sample quality is assessed separately by listening and inspecting free-running outputs.

**How to run:** In Google Colab select `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`. Output values must be produced by running the notebook; no results are pre-filled.

## 1 · Environment, reproducibility and library installation

In [ ]:
%pip -q install pretty_midi

import os, time, random, copy, zipfile, urllib.request, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Audio, Markdown
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import pretty_midi
from sklearn.metrics import accuracy_score, f1_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('PyTorch:', torch.__version__, '| Device:', DEVICE)
if DEVICE.type == 'cuda': print('GPU:', torch.cuda.get_device_name(0))

## 2 · Download the official real MAESTRO MIDI dataset

The v3 MIDI archive has an official SHA256 checksum. This cell downloads the *actual* recordings, verifies the file hash, and extracts the metadata and MIDI files. The download requires Colab internet access.

In [ ]:
URL = 'https://storage.googleapis.com/magentadata/datasets/maestro/v3.0.0/maestro-v3.0.0-midi.zip'
EXPECTED_SHA256 = '70470ee253295c8d2c71e6d9d4a815189e35c89624b76d22fce5a019d5dde12c'
ROOT = Path('/content/maestro_data') if Path('/content').exists() else Path('./maestro_data')
ROOT.mkdir(parents=True, exist_ok=True)
ZIP_PATH = ROOT / 'maestro-v3.0.0-midi.zip'

if not ZIP_PATH.exists():
    print('Downloading official MAESTRO MIDI archive (~56 MB)...')
    urllib.request.urlretrieve(URL, ZIP_PATH)
else:
    print('Using existing archive:', ZIP_PATH)

sha = hashlib.sha256()
with open(ZIP_PATH, 'rb') as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b''):
        sha.update(chunk)
if sha.hexdigest() != EXPECTED_SHA256:
    ZIP_PATH.unlink(missing_ok=True)
    raise RuntimeError('Dataset SHA256 verification failed. Archive removed; rerun to download again.')
print('Official dataset SHA256: VERIFIED')

OUT = ROOT / 'extracted'
OUT.mkdir(exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    # Standard MAESTRO archive contains the maestro-v3.0.0 root directory
    if not list(OUT.rglob('maestro-v3.0.0.csv')):
        print('Extracting MIDI files...')
        z.extractall(OUT)
metadata_paths = list(OUT.rglob('maestro-v3.0.0.csv'))
assert len(metadata_paths) == 1, 'Cannot locate MAESTRO metadata after extraction.'
BASE = metadata_paths[0].parent
meta = pd.read_csv(metadata_paths[0])
assert {'split', 'midi_filename', 'canonical_composer'}.issubset(meta.columns)
print('Official files:', len(meta))
print(meta['split'].value_counts().to_string())
print('Sample:', meta[['canonical_composer','canonical_title','split']].iloc[0].to_dict())

## 3 · Extract one-to-many examples from **real MIDI performances**

Each training example consists of **one starting piano note** and the **32 following note-on pitches** from the *same* real performance. To keep the Colab experiment manageable, we use a fixed reproducible subset of *performances* from each official split, with a limited number of windows per performance. No train/test windows are drawn from the same composition because the official metadata separates compositions.

Pitch vocabulary: 88 standard piano pitches (MIDI 21–108). We retain onset order but ignore simultaneous-note timing and other MIDI attributes in this baseline. The exact extracted sample counts are reported after running.

In [ ]:
SEQ_LEN = 32
MIN_PITCH, MAX_PITCH = 21, 108
VOCAB = MAX_PITCH - MIN_PITCH + 1  # 88 classes
EMBED = 48
HIDDEN = 96
BATCH_SIZE = 96
EPOCHS = 10
LR = 0.001

# Colab-friendly subset. Set to None to process all performances (slower).
MAX_FILES = {'train': 160, 'validation': 40, 'test': 40}
WINDOWS_PER_FILE = 32

def midi_pitch_stream(path):
    midi = pretty_midi.PrettyMIDI(str(path))
    notes = []
    for instrument in midi.instruments:
        if instrument.is_drum: continue
        for note in instrument.notes:
            if MIN_PITCH <= note.pitch <= MAX_PITCH:
                notes.append((note.start, note.pitch))
    # Sort by onset, then pitch for deterministic ordering of simultaneous notes
    notes.sort(key=lambda z: (z[0], z[1]))
    return np.asarray([p - MIN_PITCH for _, p in notes], dtype=np.int64)

def extract_split(split):
    rows = meta[meta['split'] == split].sort_values('midi_filename')
    if MAX_FILES[split] is not None:
        rows = rows.head(MAX_FILES[split])
    starts, continuations, file_ids = [], [], []
    failures = 0
    for item in rows.itertuples(index=False):
        filepath = BASE / item.midi_filename
        try:
            pitches = midi_pitch_stream(filepath)
        except Exception as exc:
            failures += 1
            continue
        possible = len(pitches) - SEQ_LEN
        if possible <= 0: continue
        starts_at = np.unique(np.linspace(0, possible-1, num=min(WINDOWS_PER_FILE, possible), dtype=int))
        for pos in starts_at:
            starts.append(pitches[pos])
            continuations.append(pitches[pos+1: pos+1+SEQ_LEN])
            file_ids.append(item.midi_filename)
    X = np.asarray(starts, dtype=np.int64)
    Y = np.asarray(continuations, dtype=np.int64)
    assert len(X) > 0 and Y.shape[1] == SEQ_LEN, f'Empty or malformed {split} split.'
    print(f'{split:10s}: {len(X):5d} real-data windows from {len(set(file_ids)):3d} MIDI performances; parse failures={failures}')
    return X, Y, set(file_ids)

X_train, Y_train, train_files = extract_split('train')
X_val, Y_val, val_files = extract_split('validation')
X_test, Y_test, test_files = extract_split('test')
assert train_files.isdisjoint(val_files | test_files)
assert val_files.isdisjoint(test_files)
assert Y_train.min() >= 0 and Y_train.max() < VOCAB
print('\nInput example: MIDI note', int(X_train[0]) + MIN_PITCH)
print('Target: next', SEQ_LEN, 'real note pitches:', (Y_train[0][:12] + MIN_PITCH).tolist(), '...')

fig, ax = plt.subplots(figsize=(12, 3.5))
ax.step(np.arange(SEQ_LEN+1), np.r_[X_train[0],Y_train[0]]+MIN_PITCH, where='mid')
ax.set(xlabel='Note-on event index', ylabel='MIDI pitch', title='Actual MAESTRO note sequence (seed + 32 next notes)')
ax.grid(alpha=.25)
plt.show()

## 4 · Data loaders + simple baseline

The task is **next-note classification with teacher forcing** during training. At each decoder position the previous correct pitch is supplied. This makes evaluation meaningful for real music; it is *not* the same as asking for the whole future sequence from the seed without feedback.

In [ ]:
def make_loader(x, y, shuffle=False):
    return DataLoader(TensorDataset(torch.from_numpy(x), torch.from_numpy(y)),
                      batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=0)
train_loader = make_loader(X_train, Y_train, True)
val_loader = make_loader(X_val, Y_val)
test_loader = make_loader(X_test, Y_test)

# Under teacher forcing, a very simple baseline is to repeat the previous true note.
prev_test = np.concatenate([X_test[:, None], Y_test[:, :-1]], axis=1)
repeat_accuracy = (prev_test == Y_test).mean() * 100
print(f'Repeat-previous-note teacher-forced test accuracy baseline: {repeat_accuracy:.2f}%')

## 5 · Three comparable *autoregressive* decoders

The decoder has two inputs *during training*: (a) the single starting-note condition and (b) the shifted true note sequence for teacher forcing. During generation, only the starting note comes from the user, and subsequent decoder tokens are **its own previous predictions**.

- **LSTM:** gated recurrence with separate hidden and cell states.
- **GRU:** gated recurrence without separate cell state.
- **TCN:** causal dilated 1D convolutions with residual blocks.

All models predict a distribution over **88 real piano pitches** at each position. They use the same training and test windows; parameter counts differ and are reported.

In [ ]:
class RecurrentDecoder(nn.Module):
    def __init__(self, cell='LSTM'):
        super().__init__()
        self.pitch_embed = nn.Embedding(VOCAB, EMBED)
        self.seed_embed = nn.Embedding(VOCAB, EMBED)
        klass = nn.LSTM if cell == 'LSTM' else nn.GRU
        self.rnn = klass(EMBED * 2, HIDDEN, num_layers=2, batch_first=True)
        self.out = nn.Linear(HIDDEN, VOCAB)

    def forward(self, seed, prev_notes):
        # seed: [B]; prev_notes: [B,T] (seed then true/generated preceding tokens)
        p = self.pitch_embed(prev_notes)
        s = self.seed_embed(seed)[:,None,:].expand(-1, prev_notes.size(1), -1)
        h, _ = self.rnn(torch.cat([p,s], dim=-1))
        return self.out(h)  # [B,T,88]

class CausalConv(nn.Module):
    def __init__(self, in_channels, out_channels, dilation):
        super().__init__()
        self.pad = 2 * dilation   # kernel_size=3
        self.conv = nn.Conv1d(in_channels, out_channels, 3,
                              dilation=dilation, padding=self.pad)
    def forward(self,x):
        y = self.conv(x)
        return y[:,:,:-self.pad] if self.pad else y

class TCNBlock(nn.Module):
    def __init__(self, inc, outc, dilation):
        super().__init__()
        self.main = nn.Sequential(CausalConv(inc,outc,dilation), nn.ReLU(),
                                  nn.Dropout(0.1), CausalConv(outc,outc,dilation),
                                  nn.ReLU(), nn.Dropout(0.1))
        self.skip = nn.Conv1d(inc,outc,1) if inc != outc else nn.Identity()
    def forward(self,x): return torch.relu(self.main(x) + self.skip(x))

class TCNDecoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.pitch_embed = nn.Embedding(VOCAB, EMBED)
        self.seed_embed = nn.Embedding(VOCAB, EMBED)
        self.stack = nn.Sequential(TCNBlock(EMBED*2,HIDDEN,1),
                                   TCNBlock(HIDDEN,HIDDEN,2),
                                   TCNBlock(HIDDEN,HIDDEN,4),
                                   TCNBlock(HIDDEN,HIDDEN,8))
        self.out = nn.Linear(HIDDEN,VOCAB)
    def forward(self, seed, prev_notes):
        p = self.pitch_embed(prev_notes)
        s = self.seed_embed(seed)[:,None,:].expand(-1,prev_notes.size(1),-1)
        z = self.stack(torch.cat([p,s],-1).transpose(1,2)).transpose(1,2)
        return self.out(z)

def make_models():
    return {'LSTM': RecurrentDecoder('LSTM').to(DEVICE),
            'GRU': RecurrentDecoder('GRU').to(DEVICE),
            'TCN': TCNDecoder().to(DEVICE)}
models = make_models()
with torch.no_grad():
    demo_seed = torch.tensor([30,45], device=DEVICE)
    demo_prev = demo_seed[:,None].expand(2,SEQ_LEN)
    for name,m in models.items():
        m.eval()
        shape = tuple(m(demo_seed,demo_prev).shape)
        params = sum(p.numel() for p in m.parameters())
        print(name, 'output:', shape, 'parameters:', f'{params:,}')
        assert shape == (2, SEQ_LEN, VOCAB)

## 6 · Train with validation and save the best checkpoint

For fairness, all three models use Adam, the same batch size, epochs, data windows and the same validation-based checkpoint criterion. We measure wall-clock training time (including validation). **No fabricated accuracy values are supplied**.

In [ ]:
loss_fn = nn.CrossEntropyLoss()

def step_inputs(seed, target):
    # Teacher forcing: previous true note is visible at every prediction position.
    return torch.cat([seed[:,None], target[:,:-1]], dim=1)

def evaluate(model, loader):
    model.eval()
    total_ce, correct, top5, count = 0., 0, 0, 0
    with torch.inference_mode():
        for seed, target in loader:
            seed, target = seed.to(DEVICE), target.to(DEVICE)
            logits = model(seed, step_inputs(seed,target))
            total_ce += loss_fn(logits.reshape(-1,VOCAB), target.reshape(-1)).item()*target.numel()
            correct += (logits.argmax(-1)==target).sum().item()
            top5 += (logits.topk(5,dim=-1).indices==target.unsqueeze(-1)).any(-1).sum().item()
            count += target.numel()
    return {'loss':total_ce/count, 'accuracy':100*correct/count, 'top5':100*top5/count}

def fit(model, name):
    optim = torch.optim.Adam(model.parameters(),lr=LR)
    hist=[]; best_loss=float('inf'); best=None
    if DEVICE.type=='cuda': torch.cuda.synchronize()
    t0=time.perf_counter()
    for ep in range(1,EPOCHS+1):
        model.train(); total=0.; n=0
        for seed,target in train_loader:
            seed,target=seed.to(DEVICE),target.to(DEVICE)
            optim.zero_grad(set_to_none=True)
            logits = model(seed,step_inputs(seed,target))
            loss = loss_fn(logits.reshape(-1,VOCAB),target.reshape(-1))
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optim.step()
            total += loss.item()*target.numel(); n += target.numel()
        val = evaluate(model,val_loader)
        hist.append({'epoch':ep,'train_loss':total/n,
                     'val_loss':val['loss'],'val_accuracy':val['accuracy']})
        if val['loss'] < best_loss:
            best_loss=val['loss']
            best={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        print(f'{name:4s} epoch {ep:02d}/{EPOCHS}: train_loss={total/n:.3f} '
              f'val_loss={val["loss"]:.3f} val_acc={val["accuracy"]:.2f}%')
    if DEVICE.type=='cuda': torch.cuda.synchronize()
    elapsed=time.perf_counter()-t0
    model.load_state_dict(best)
    return pd.DataFrame(hist),elapsed

histories={}; training_times={}
for name,model in models.items():
    print('\nTRAINING',name)
    h,elapsed=fit(model,name)
    histories[name]=h
    training_times[name]=elapsed
    print(f'{name} total training+validation time: {elapsed:.1f}s')
print('\nFinished training and restored each model’s best-validation-loss checkpoint.')

## 7 · Final held-out test metrics and honest comparisons

These are **teacher-forced next-note metrics** on previously unseen MAESTRO compositions, **not** a claim that every fully generated 32-note melody matches a reference. Top-1, top-5, macro-F1, perplexity, model parameters, and timing complement listening to free-running generations.

In [ ]:
rows=[]
for name,model in models.items():
    test=evaluate(model,test_loader)
    preds=[]; gold=[]
    model.eval()
    if DEVICE.type=='cuda':torch.cuda.synchronize()
    sample_batch=next(iter(test_loader))[0].to(DEVICE)
    fake_prev=sample_batch[:,None].expand(-1,SEQ_LEN)
    with torch.inference_mode():
        for _ in range(10):model(sample_batch,fake_prev)
        if DEVICE.type=='cuda':torch.cuda.synchronize()
        t0=time.perf_counter()
        for _ in range(30):model(sample_batch,fake_prev)
        if DEVICE.type=='cuda':torch.cuda.synchronize()
        ms=1000*(time.perf_counter()-t0)/30
        for seed,target in test_loader:
            seed=seed.to(DEVICE); target=target.to(DEVICE)
            pred=model(seed,step_inputs(seed,target)).argmax(-1)
            preds.append(pred.cpu().numpy());gold.append(target.cpu().numpy())
    pred=np.concatenate(preds).ravel();gt=np.concatenate(gold).ravel()
    rows.append({'Model':name,'Top-1 Accuracy (%)':test['accuracy'],
                 'Top-5 Accuracy (%)':test['top5'],
                 'Macro F1 (%)':100*f1_score(gt,pred,labels=np.arange(VOCAB),average='macro',zero_division=0),
                 'Test Cross-Entropy':test['loss'],
                 'Perplexity':float(np.exp(min(test['loss'],50))),
                 'Parameters':sum(p.numel() for p in model.parameters()),
                 'Train+Validation (s)':training_times[name],
                 'Forward Latency (ms/batch)':ms})
results=pd.DataFrame(rows).sort_values('Top-1 Accuracy (%)',ascending=False)
display(results.round(3))
print(f'Baseline repeat-previous-note accuracy: {repeat_accuracy:.2f}%')
results.to_csv('MAESTRO_LSTM_GRU_TCN_results.csv',index=False)

## 8 · Presentation-ready visualizations

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(13,4.5))
for name,h in histories.items():
    axes[0].plot(h['epoch'],h['train_loss'],label=f'{name} train')
    axes[0].plot(h['epoch'],h['val_loss'],'--',label=f'{name} validation')
    axes[1].plot(h['epoch'],h['val_accuracy'],label=name)
axes[0].set(xlabel='Epoch',ylabel='Cross-entropy',title='Train vs validation loss')
axes[1].set(xlabel='Epoch',ylabel='Note accuracy (%)',title='Validation next-note accuracy')
for ax in axes: ax.grid(alpha=.25);ax.legend()
plt.tight_layout();plt.savefig('MAESTRO_learning_curves.png',dpi=180,bbox_inches='tight');plt.show()

fig,axs=plt.subplots(1,3,figsize=(14,3.8))
for ax,col,title in zip(axs,['Top-1 Accuracy (%)','Macro F1 (%)','Train+Validation (s)'],
                         ['Test top-1 accuracy','Test macro F1','Training + validation time']):
    ax.bar(results.Model,results[col]);ax.set_title(title);ax.set_ylabel(col);ax.grid(axis='y',alpha=.2)
plt.tight_layout();plt.savefig('MAESTRO_comparison.png',dpi=180,bbox_inches='tight');plt.show()

## 9 · Genuine one-to-many generation from **one** MIDI note

**Critical distinction:** At inference, no ground-truth continuation is provided. The decoder repeatedly predicts a next note and then feeds that generated note back into itself. All three models start from the **exact same one-note seed**. We can sample from a top-*k* distribution (more varied) or choose argmax (more deterministic).

In [ ]:
@torch.inference_mode()
def generate(model, seed_midi=60, n_notes=SEQ_LEN, temperature=0.9, top_k=8, use_sampling=True, rng_seed=7):
    assert MIN_PITCH <= seed_midi <= MAX_PITCH
    assert temperature>0
    model.eval()
    seed=torch.tensor([seed_midi-MIN_PITCH],device=DEVICE,dtype=torch.long)
    prev=seed[:,None]
    output=[]
    generator=torch.Generator(device=DEVICE.type).manual_seed(rng_seed)
    for _ in range(n_notes):
        logits=model(seed,prev)[:,-1,:] / temperature
        if use_sampling:
            k=min(top_k,VOCAB)
            vals,idx=logits.topk(k,dim=-1)
            probs=torch.softmax(vals,dim=-1)
            choice=torch.multinomial(probs,num_samples=1,generator=generator)
            next_token=idx.gather(1,choice)
        else:
            next_token=logits.argmax(-1,keepdim=True)
        output.append(int(next_token.item())+MIN_PITCH)
        prev=torch.cat([prev,next_token],dim=1)
    return output

SEED_MIDI=60  # C4, a genuine single-note input, not a synthetic genre label
samples={name:generate(model,SEED_MIDI) for name,model in models.items()}
print('ONE INPUT:',pretty_midi.note_number_to_name(SEED_MIDI),'(MIDI',SEED_MIDI,')')
for name,notes in samples.items():
    print(name,':',' '.join(pretty_midi.note_number_to_name(p) for p in notes[:20]),'...')

fig,ax=plt.subplots(figsize=(14,4.5))
for name,notes in samples.items():
    ax.plot(np.arange(1,len(notes)+1),notes,marker='.',label=name,alpha=.8)
ax.axhline(SEED_MIDI,ls=':',color='black',label='Seed pitch')
ax.set(xlabel='Generated note number',ylabel='MIDI pitch',title='One real starting note → 32 free-running generated pitches')
ax.legend();ax.grid(alpha=.2)
plt.tight_layout();plt.savefig('MAESTRO_generated_notes.png',dpi=180,bbox_inches='tight');plt.show()

## 10 · Listen to the generated notes (simple synthesized sound)

This is **not original MAESTRO audio** and does not reproduce expressive timing/dynamics. It turns model-predicted MIDI pitches into simple piano-like tones to make the output easy to demonstrate.

In [ ]:
def make_audio(midi_notes, sec_per_note=0.18, sr=22050):
    parts=[]
    for pitch in midi_notes:
        f=pretty_midi.note_number_to_hz(int(pitch))
        t=np.arange(int(sr*sec_per_note))/sr
        tone=(np.sin(2*np.pi*f*t) + 0.22*np.sin(4*np.pi*f*t) + 0.08*np.sin(6*np.pi*f*t))
        env=(1-np.exp(-t*65))*np.exp(-t*7)
        parts.append((tone*env).astype(np.float32))
    wave=np.concatenate(parts)
    wave/=max(1.,np.max(np.abs(wave)))
    return wave,sr

for name,seq in samples.items():
    display(Markdown(f'**{name}: generated from a single C4 seed**'))
    wave,sr=make_audio([SEED_MIDI]+seq)
    display(Audio(wave,rate=sr))

## 11 · Interactive demonstration (optional)

Select **one starting MIDI note** and a model. Clicking **Generate** creates a brand-new 32-note continuation. The model uses its own prior predicted notes as feedback.

In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output
pitch_widget=widgets.Dropdown(options=[(pretty_midi.note_number_to_name(p)+f' ({p})',p) for p in range(48,85)],value=60,description='Seed note')
model_widget=widgets.Dropdown(options=list(models),value='LSTM',description='Decoder')
seed_widget=widgets.IntSlider(value=7,min=0,max=100,description='Sample seed')
button=widgets.Button(description='Generate 32 notes',button_style='success')
output_box=widgets.Output()
def on_generate(_):
    with output_box:
        clear_output(wait=True)
        midi_note=pitch_widget.value
        generated=generate(models[model_widget.value],midi_note,rng_seed=seed_widget.value)
        print('One input:',pretty_midi.note_number_to_name(midi_note),'| Decoder:',model_widget.value)
        print('Generated notes:', ' '.join(pretty_midi.note_number_to_name(n) for n in generated))
        audio,sr=make_audio([midi_note]+generated)
        display(Audio(audio,rate=sr))
button.on_click(on_generate)
display(widgets.VBox([pitch_widget,model_widget,seed_widget,button,output_box]))

## 12 · Save results and model weights

After running the notebook, these are genuine files containing *your executed experimental results*. You can download them from the Colab left-hand Files pane.

In [ ]:
for name,model in models.items():
    torch.save(model.state_dict(),f'MAESTRO_{name}_decoder.pth')
print('Saved: MAESTRO_LSTM_GRU_TCN_results.csv, three .pth weights, and three .png figures')
print('Tested on official MAESTRO test compositions; outputs above are produced by this run.')

## 13 · Viva / demonstration notes

**Problem statement:** Compare an LSTM decoder, a GRU decoder, and a TCN decoder for one-to-many piano pitch continuation on the **real MAESTRO v3 MIDI dataset**.

**Exactly one input at inference:** one starting MIDI piano pitch (e.g., **60 = C4**). **Output:** 32 generated pitches, one after another.

**How is it one-to-many?** One supplied seed conditions a sequence of 32 output notes. During training, teacher forcing uses the true earlier notes to learn next-note prediction; during generation, only the seed is externally supplied.

**Encoder–decoder?** The starting-note embedding acts as a learned conditioning representation. LSTM/GRU/TCN act as *autoregressive sequence decoders*. There is no separate sequence encoder or attention layer.

**Why might accuracy be low?** A single first note does not determine the next 32 pitches of a real composition. Test accuracy is next-note accuracy *given the actual preceding notes*, not exact generated-melody match. Music can have several valid continuations.

**What to show ma'am:** verified MAESTRO download → official split counts → real notes → parameter counts → loss and comparison plots → live generation and audio → limitations.

**What this does NOT claim:** it does not model durations, note overlaps, performer expression, audio waveforms, or complete piano compositions. It does not contain precomputed/fabricated results.

**Dataset credit:** Hawthorne et al., *Enabling Factorized Piano Music Modeling and Generation with the MAESTRO Dataset* (ICLR 2019). Dataset: https://magenta.tensorflow.org/datasets/maestro . Licensed CC BY-NC-SA 4.0. The official metadata splits prevent the same composition from crossing subsets.